# LEA Rollator Center of Mass Calculation #

## Data Analysis of LEA Rollator Force Plates Measurements ##

### Imports ###

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.float_format', '{:.2f}'.format)

file_paths = {
    "measurement_1_rollator_only": "Lea_0002_2026_05_22_142524-convert.xlsx",
    "measurement_2_user_plus_rollator_2plates": "Lea_0003_2026_05_22_143422-convert.xlsx",
    "measurement_3_user_plus_rollator_1plate": "Lea_0004_2026_05_22_144123-convert.xlsx",
    "measurement_4_rollator_flat_compression": "Lea_0005_2026_05_22_150246-convert.xlsx"
}

dfs = {}

for name, path in file_paths.items():
    try:
        df = pd.read_excel(path)
        dfs[name] = df
        print(f"Loaded {name}: shape = {df.shape}")
    except Exception as e:
        print(f"Failed to load {name}: {e}")

Loaded measurement_1_rollator_only: shape = (2680, 22)
Loaded measurement_2_user_plus_rollator_2plates: shape = (17595, 22)
Loaded measurement_3_user_plus_rollator_1plate: shape = (13592, 22)
Loaded measurement_4_rollator_flat_compression: shape = (4133, 22)


### Elementary Data Analysis ###

In [ ]:
stats = {}

for name, df in dfs.items():
    
    numeric_df = df.apply(pd.to_numeric, errors='coerce')
    
    # Compute stats
    stats[name] = pd.DataFrame({
        "mean": numeric_df.mean(),
        "variance": numeric_df.var(),
        "std": numeric_df.std(),
        "min": numeric_df.min(),
        "max": numeric_df.max()
    })

    print(f"Processed stats for {name}")

#TOGGLE ON AND OFF THE STATISTICS!!

display(stats["measurement_1_rollator_only"])
# display(stats["measurement_2_user_plus_rollator_2plates"])
# display(stats["measurement_3_user_plus_rollator_1plate"])
#display(stats["measurement_4_rollator_flat_compression"])

Processed stats for measurement_1_rollator_only
Processed stats for measurement_2_user_plus_rollator_2plates
Processed stats for measurement_3_user_plus_rollator_1plate
Processed stats for measurement_4_rollator_flat_compression


,mean,variance,std,min,max
1:EventCounter,387356.50,598756.67,773.79,386017.00,388696.00
1:Sync,0.00,0.00,0.00,0.00,0.00
1:Aux,255.00,0.00,0.00,255.00,255.00
1:Fx,0.67,0.13,0.36,-0.38,1.59
1:Fy,1.91,0.09,0.30,1.01,2.78
1:Fz,257.91,0.06,0.24,256.73,258.58
1:Mx,-0.90,0.00,0.06,-1.05,-0.70
1:My,10.28,0.00,0.07,10.03,10.49
1:Mz,-0.66,0.01,0.12,-1.07,-0.23
1:COPx,-0.04,0.00,0.00,-0.04,-0.04


,mean,variance,std,min,max
1:EventCounter,2630091.00,1423818.50,1193.24,2628025.00,2632157.00
1:Sync,0.00,0.00,0.00,0.00,0.00
1:Aux,255.00,0.00,0.00,255.00,255.00
1:Fx,0.52,0.14,0.38,-0.45,1.51
1:Fy,-19.09,0.07,0.27,-19.82,-18.45
1:Fz,59.64,0.07,0.26,58.59,60.42
1:Mx,-10.38,0.00,0.06,-10.61,-10.17
1:My,2.14,0.01,0.07,1.88,2.42
1:Mz,0.69,0.01,0.11,0.25,1.05
1:COPx,-0.04,0.00,0.00,-0.04,-0.03


### Feature Selection ###

Features (for each force plate): 

- EventCounter
- Sync
- Aux
- Fx
- Fy
- Fz
- Mx
- My
- Mz
- CoPx
- CoPy

**Feature elimination:** 

The EventCounter is only the time stamp and it is therefore unnecessary for calculations.

The measured features Sync and Aux have a variance of zero. They will therefore be removed.

# Calculating CoM #

### X and Y CoM using Measurement 1 ###

In [ ]:
y_com = 0  # mm (by symmetry assumption)

df = dfs["measurement_1_rollator_only"]

# mean vertical forces
Fz1 = df["1:Fz"].mean()  # rear
Fz2 = df["2:Fz"].mean()  # front

# geometry
L_wheelbase = 430  # mm (given)
gap = 10           # mm
L = L_wheelbase
W = Fz1 + Fz2

# CoM calculation
x_com = (Fz2 * L) / (W)

print("X-CoM (mm from front axle):", x_com)
print("Y-CoM (mm from center):", y_com)

X-CoM (mm from rear axle): 175.4907241975882
Y-CoM (mm from rear axle): 0


### Z Coordinate CoM using Measurement 4 ###

In [13]:
# Measurement 4
df = dfs["measurement_4_rollator_flat_compression"]

Width_ForcePlate = 0.5   # m
gap = 0.01                 # m

Fz1 = df["1:Fz"].mean()
Fz2 = df["2:Fz"].mean()

COPy1 = df["1:COPy"].mean()
COPy2 = df["2:COPy"].mean()

y10 = Width_ForcePlate*1.5 + gap
y20 = Width_ForcePlate/2

y1 = y10 + COPy1
y2 = y20 + COPy2


y_center = (Fz1 * y1 + Fz2 * y2) / (Fz1 + Fz2)


print(f"Fz1 = {Fz1:.2f} N")
print(f"Fz2 = {Fz2:.2f} N")

print(f"COPy1 = {COPy1:.2f} m")
print(f"COPy2 = {COPy2:.2f} m")

print(f"y1 = {y1:.2f} m")
print(f"y2 = {y2:.2f} m")

print(f"\nCoM_z = {y_center:.2f} m")




Fz1 = 59.64 N
Fz2 = 378.94 N
COPy1 = -0.17 m
COPy2 = -0.05 m
y1 = 0.59 m
y2 = 0.20 m

CoM_z = 0.25 m


### Rollator and User Combined

Measurement 3 contains the forces of just the rollator on one force plate and then the user on the other.

In [ ]:
# Measurement 3: User + Rollator
# Combined CoM from force-weighted COP

df = dfs["measurement_3_user_plus_rollator_1plate"]

# Only use data after user starts interacting
df_after = df[df["1:EventCounter"] > 1353484]

Fz1 = df_after["1:Fz"].mean()
Fz2 = df_after["2:Fz"].mean()

COPx1 = df_after["1:COPx"].mean()
COPy1 = df_after["1:COPy"].mean()

COPx2 = df_after["2:COPx"].mean()
COPy2 = df_after["2:COPy"].mean()


y10 = Width_ForcePlate*1.5 + gap
y20 = Width_ForcePlate/2

y1 = y10 + COPy1
y2 = y20 + COPy2


y_center = (Fz1*y1+Fz2*y2)/(Fz1+Fz2)


print(f"Fz1 = {Fz1:.2f} N")
print(f"Fz2 = {Fz2:.2f} N")

print(f"COPy1 = {COPy1:.2f} m")
print(f"COPy2 = {COPy2:.2f} m")

print(f"y1 = {y1:.2f} m")
print(f"y2 = {y2:.2f} m")

print(f"\nCoM_y = {y_center:.2f} m")
print("measured from the end of Force Plate 2")



Fz1 = 445.36 N
Fz2 = 555.07 N
COPy1 = -0.04 m
COPy2 = -0.12 m
y1 = 0.72 m
y2 = 0.13 m

CoM_y = 0.39 m
measured from the end of Force Plate 2
